# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² CRC Survivors dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is defined by a Croissant schema package accessible at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

We start by loading the dataset metadata using the `mlcroissant` library. This allows us to access the dataset structure and its field definitions, as well as records for further analysis.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

List available record sets, their fields, columns, and corresponding `@id`s. These IDs are used to reference entities consistently within the dataset.

In [ ]:
# Helper functions to pretty print
def print_record_sets(dataset):
    print("Available Record Sets:")
    for rs in dataset.record_sets():
        print(f"- Record Set: {rs['name']} | @id: {rs['@id']}")

def print_fields_for_record_set(dataset, record_set_id):
    print(f"\nFields for Record Set @id={record_set_id}:")
    fields = dataset.fields(record_set=record_set_id)
    for f in fields:
        print(f"  Field: {f['name']} | @id: {f['@id']} | dataType: {f.get('dataType')}")
        for c in f.get('column', []):
            print(f"    -> Column @id: {c['@id']} | Path: {c.get('path', '')}")

# Print available record sets and fields
print_record_sets(dataset)

# For demonstration, pick the first record set and print its fields
record_sets = [rs['@id'] for rs in dataset.record_sets()]
if record_sets:
    # We'll focus on the first record set for the rest of the notebook
    main_record_set_id = record_sets[0]
    print_fields_for_record_set(dataset, main_record_set_id)
else:
    print("No record sets found in the dataset.")

## 3. Data Extraction

Load data for the available record sets into Pandas DataFrames for analysis using their `@id`s, as displayed above.

In [ ]:
# Load all record sets into DataFrames (referenced by their @id)
dataframes = {}
for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for Record Set @id={record_set_id}")
    else:
        print(f"No records found for Record Set @id={record_set_id}")

# Display columns for the main record set
if record_sets:
    print(f"\nColumns in main record set (@id={main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Select numeric and categorical fields by their `@id` (refer to output above), filter and normalize data, and group by non-numeric attributes for further aggregation or insight.

In [ ]:
# Example: Analyze 'Age' field, which is commonly present in clinical datasets
# Replace <numeric_field_id> and <group_field_id> with the actual @id per Section 2, e.g., 'age', 'sex', etc.
numeric_field_id = None
group_field_id = None

# Try to guess common field IDs for this type of medical dataset
possible_age_fields = ['age', 'Age', 'schema:age', 'cr:age', 'https://schema.org/age']
possible_sex_fields = ['sex', 'Sex', 'schema:sex', 'cr:sex', 'https://schema.org/sex', 'gender']

main_df = dataframes.get(main_record_set_id)
if main_df is not None:
    # Attempt to find a numeric field for demonstration
    for col in main_df.columns:
        if any(x in col for x in possible_age_fields):
            numeric_field_id = col
            break
    # Attempt to find a group-by field
    for col in main_df.columns:
        if any(x in col for x in possible_sex_fields):
            group_field_id = col
            break

    # Check if found, if not, prompt user
    print(f"Using numeric_field_id: {numeric_field_id}, group_field_id: {group_field_id}\n")
    
    # If a numeric field is found, apply EDA
    if numeric_field_id and numeric_field_id in main_df.columns:
        # Ensure the data is numeric
        main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
        threshold = main_df[numeric_field_id].mean() if not main_df[numeric_field_id].isnull().all() else 0
        filtered_df = main_df[main_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > mean ({threshold}):")
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())
    else:
        print("No numeric field detected for EDA.")

    # Group by a categorical field, if found
    if group_field_id and group_field_id in main_df.columns and numeric_field_id and numeric_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped data by {group_field_id}, mean of {numeric_field_id}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found to group by.")
else:
    print("Main DataFrame is not available.")

## 5. Visualization

Visualize data distributions or field relationships. You can create histograms, boxplots, or bar plots for relevant numeric/categorical fields. Example below uses `matplotlib` and `seaborn` if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field_id:
    fig, axs = plt.subplots(1,2, figsize=(12,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True, ax=axs[0])
    axs[0].set_title(f"Distribution of {numeric_field_id}")
    if group_field_id and group_field_id in main_df.columns:
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id], ax=axs[1])
        axs[1].set_title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()
else:
    print("Cannot plot: numeric or group field not detected.")

## 6. Conclusion

In this notebook, we demonstrated how to load and explore a clinical dataset described by a Croissant schema using `mlcroissant`. We:
- Loaded dataset metadata and record set structure.
- Explored available record sets and their fields by their `@id`s.
- Loaded data for each record set and demonstrated exploratory analysis, including filtering and grouping using `@id` references.
- Visualized relevant numeric and categorical data fields.

This workflow can be adapted to any Croissant-structured dataset for efficient, reproducible, and FAIR data analysis.